In [15]:
import subprocess
import time

def saga_polygonize(input_raster_sgrd: str, output_vector_shp: str):
    start_time = time.time()
    cmd = [
        "saga_cmd",
        "shapes_grid",
        "14",  # Polygonize module ID in shapes_grid library
        "-INPUT", input_raster_sgrd,
        "-POLYGONS", output_vector_shp
    ]

    try:
        subprocess.run(cmd, check=True)
        elapsed = time.time() - start_time
        days = int(elapsed // 86400)
        hours = int((elapsed % 86400) // 3600)
        minutes = int((elapsed % 3600) // 60)
        seconds = int(elapsed % 60)
        print(f"Polygonize completed: {output_vector_shp}")
        print(f"Time taken: {days}d {hours}h {minutes}m {seconds}s")
    except subprocess.CalledProcessError as e:
        print(f"Error running SAGA polygonize: {e}")

# Example usage:
# First convert GeoTIFF to SAGA grid:
# saga_cmd io_gdal 0 -FILES your_raster.tif -GRIDS your_raster.sgrd
# Then polygonize:
# saga_polygonize("your_raster.sgrd", "output_vector.shp")


In [16]:
import os 
interp_dir = 'interp_dir'
os.makedirs(interp_dir, exist_ok=True)
dem_fn = '/home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif'
#------------------
shp_fn = os.path.join(interp_dir, os.path.basename(dem_fn).replace('.tif','.shp'))
shp_fn

'interp_dir/N13E103_edem_egm_3C.shp'

In [17]:
saga_polygonize(dem_fn, shp_fn)

____________________________

   #####   ##   #####    ##
  ###     ###  ##       ###
   ###   # ## ##  #### # ##
    ### ##### ##    # #####
 ##### #   ##  ##### #   ##
____________________________

SAGA Version: 8.2.2


Library:	Shapes-Grid Tools
Category:	Shapes
File:	/usr/lib/x86_64-linux-gnu/saga/libshapes_grid.so
Description:
Tools related to gridded and vector data (conversions, combinations, etc.).

Tools:
 [0]	Add Grid Values to Points
 [1]	Add Grid Values to Shapes
 [2]	Grid Statistics for Polygons
 [3]	Grid Values to Points
 [4]	Grid Values to Points (randomly)
 [5]	Contour Lines from Grid
 [6]	Vectorising Grid Classes
 [7]	Clip Grid with Polygon
 [8]	Grid Statistics for Points
 [9]	Local Minima and Maxima
 [10]	Grid System Extent
 [11]	Clip Grid with Rectangle
 [15]	Gradient Vectors from Surface
 [16]	Gradient Vectors from Direction and Length
 [17]	Gradient Vectors from Directional Components
 [18]	Grid Classes Area for Polygons

type -h or --help for further information



Error: select a tool


Error running SAGA polygonize: Command '['saga_cmd', 'shapes_grid', '14', '-INPUT', '/home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif', '-POLYGONS', 'interp_dir/N13E103_edem_egm_3C.shp']' died with <Signals.SIGSEGV: 11>.


In [2]:
import os
import time
import subprocess
from datetime import timedelta

def close_gaps_with_stepwise_resampling(opath, ipath, overwrite=False):
    if os.path.exists(opath) and not overwrite:
        print(f"[SKIP] close_gaps_with_stepwise_resampling → {opath} exists")
        return

    print(f"[START] close_gaps_with_stepwise_resampling")
    ti = time.time()

    cmd = f'saga_cmd grid_tools 29 -INPUT {ipath} -RESULT {opath}'
    os.system(cmd)

    tf = time.time()
    elapsed = tf - ti
    print(f"[DONE] close_gaps_with_stepwise_resampling in {elapsed:.2f}s ({timedelta(seconds=elapsed)})")

def close_gap_with_spline(opath, ipath, overwrite=False):
    if os.path.exists(opath) and not overwrite:
        print(f"[SKIP] close_gap_with_spline → {opath} exists")
        return

    print(f"[START] close_gap_with_spline")
    ti = time.time()

    cmd = f'saga_cmd grid_tools 25 -INPUT {ipath} -CLOSED {opath}'
    os.system(cmd)

    tf = time.time()
    elapsed = tf - ti
    print(f"[DONE] close_gap_with_spline in {elapsed:.2f}s ({timedelta(seconds=elapsed)})")

def close_gaps_with_tension_thresh(opath, ipath, overwrite=False):
    if os.path.exists(opath) and not overwrite:
        print(f"[SKIP] close_gaps_with_tension_thresh → {opath} exists")
        return

    print(f"[START] close_gaps_with_tension_thresh")
    ti = time.time()

    cmd = f'saga_cmd grid_tools 7 -INPUT {ipath} -RESULT {opath}'
    os.system(cmd)

    tf = time.time()
    elapsed = tf - ti
    print(f"[DONE] close_gaps_with_tension_thresh in {elapsed:.2f}s ({timedelta(seconds=elapsed)})")

def mosaicking(opath, ipath_list, overwrite=False):
    if os.path.exists(opath) and not overwrite:
        print(f"[SKIP] mosaicking → {opath} exists")
        return

    print(f"[START] mosaicking")
    ti = time.time()

    cmd = f'saga_cmd grid_tools 3 -GRIDS {ipath_list} -TARGET_OUT_GRID {opath}'
    os.system(cmd)

    tf = time.time()
    elapsed = tf - ti
    print(f"[DONE] mosaicking in {elapsed:.2f}s ({timedelta(seconds=elapsed)})")

def mosaic_grids(output, file_list, overlap=1, resampling=3, match=0, blend_dist=10.0, overwrite=False):
    if os.path.exists(output) and not overwrite:
        print(f"[SKIP] mosaic_grids → {output} exists")
        return

    print(f"[START] mosaic_grids")
    ti = time.time()

    cmd = [
        "saga_cmd", "grid_tools", "3",
        "-FILE_LIST", file_list,
        "-TARGET_OUT_GRID", output,
        "-TYPE", "7",  # Float32
        "-OVERLAP", str(overlap),
        "-RESAMPLING", str(resampling),
        "-BLEND_DIST", str(blend_dist),
        "-MATCH", str(match),
        "-TARGET_DEFINITION", "1"
    ]
    subprocess.run(cmd, check=True)

    tf = time.time()
    elapsed = tf - ti
    print(f"[DONE] mosaic_grids in {elapsed:.2f}s ({timedelta(seconds=elapsed)})")

def patching(opath, org_fn, add_fn, overwrite=False):
    if os.path.exists(opath) and not overwrite:
        print(f"[SKIP] patching → {opath} exists")
        return

    print(f"[START] patching")
    ti = time.time()

    cmd = f'saga_cmd grid_tools 5 -ORIGINAL {org_fn} -COMPLETED {opath} -ADDITIONAL {add_fn}'
    os.system(cmd)

    tf = time.time()
    elapsed = tf - ti
    print(f"[DONE] patching in {elapsed:.2f}s ({timedelta(seconds=elapsed)})")


In [1]:
import os 
interp_dir = 'interp_dir'
os.makedirs(interp_dir, exist_ok=True)
#SAGA_cmd = "/home/ljp238/Downloads/SOFTWARE/saga-9.9.0/saga-gis/src/saga_core/saga_cmd"
saga_cmd = "/home/ljp238/Downloads/SOFTWARE/saga-8-5-1/saga-gis/src/saga_core/saga_cmd/man/saga_cmd.1"
dem_fn = "/home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif"
close_gaps_fn = 'close_gaps_saga.tif'

In [3]:
close_gaps_with_stepwise_resampling(opath='cgap_stepwise_resampling.tif', 
                                    ipath=dem_fn, overwrite=False)

[START] close_gaps_with_stepwise_resampling
____________________________

   #####   ##   #####    ##
  ###     ###  ##       ###
   ###   # ## ##  #### # ##
    ### ##### ##    # #####
 ##### #   ##  ##### #   ##
____________________________

SAGA Version: 8.2.2

____________________________
library path: /usr/lib/x86_64-linux-gnu/saga/
library name: libgrid_tools
library     : grid_tools
tool        : Close Gaps with Stepwise Resampling
identifier  : 29
author      : O.Conrad (c) 2012
processors  : 56 [56]
____________________________

loading: N13E103_edem_egm_3C

100%
[Close Gaps with Stepwise Resampling] Execution started...

__________
[Close Gaps with Stepwise Resampling] Parameters:

Grid System: 0.000111; 9001x 9001y; 103x 13y
Grid: N13E103_edem_egm_3C
Mask: <not set>
Result: Result
Resampling: B-Spline Interpolation
Grow Factor: 2


100%

100%

100%

100%

100%

100%

100%

 99%

 98%

 95%

 91%

 81%

 67%

 51%

 51%

101%
__________
total execution time: 56000 millisecond

corrupted double-linked list


[DONE] close_gaps_with_stepwise_resampling in 63.65s (0:01:03.648901)


Aborted (core dumped)
